# Analyse de lambda

## 1. Imports

In [155]:
import math
import numpy as np
import matplotlib.pyplot as plt
import time
import random
import pandas as pd

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2. Paramètres et simulateur de marché

In [156]:
# Paramètres fixes (Section 5.1, p.27)
T = 1.0
dt = 1.0 / 252
n_steps = int(round(T / dt))   # 252
x0 = 1.0
z = 1.4
r_rate = 0.02

# Gardes-fous numériques minimaux
# - phi2 > 0 est structurel dans le papier
# - le clip sur log-variance évite les overflow/underflow exponentiels
PHI2_CLIP = (1e-12, 10.0)
LOG_VAR_CLIP = (-50.0, 50.0)

print(f"n_steps={n_steps}, dt={dt:.6f}")

n_steps=252, dt=0.003968


`sharpe_ratio` calcule $\rho = (\mu-r)/\sigma$. 

`compute_w_star` calcule la valeur fermée de $w^\star$. C’est le multiplicateur de Lagrange qui permet d’imposer la contrainte $E[X_T]=z$.

`simulate_gbm_prices` simule une trajectoire de prix de l’actif risqué sous un mouvement brownien géométrique, avec $\mu$ et $\sigma$ constants.

`discounted_wealth_step` met à jour la richesse actualisée entre $t$ et $t+\Delta t$. On prend la position risquée $u_t$, puis on lui applique le rendement actualisé réalisé sur ce pas de temps.

In [157]:
def sharpe_ratio(mu, r, sigma):
    # Ratio de Sharpe du modèle
    return (mu - r) / sigma

def compute_w_star(rho, T=1.0, x0=1.0, z=1.4, tol=1e-14):
    # Multiplicateur de Lagrange en forme fermée
    a = rho**2 * T
    denom = math.expm1(a)

    # Cas dégénéré : rho ≈ 0
    if abs(denom) < tol:
        if abs(z - x0) < tol:
            return np.nan
        raise ValueError("rho ≈ 0 et z != x0 : cible non atteignable.")

    e = math.exp(a)
    return (z * e - x0) / denom

def simulate_gbm_prices(mu, sigma, n_prices, dt, S0, rng):
    # Simulation d'un GBM sur la grille de temps
    eps = rng.normal(0.0, 1.0, size=n_prices - 1)
    log_ret = (mu - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * eps

    S = np.empty(n_prices, dtype=float)
    S[0] = S0
    S[1:] = S0 * np.exp(np.cumsum(log_ret))
    return S

def discounted_wealth_step(x_t, u_t, s_t, s_tp1, r, dt):
    discounted_return = math.exp(-r * dt) * (s_tp1 / s_t) - 1.0
    return x_t + u_t * discounted_return

def wealth_step_direct(x_t, u_t, mu, sigma, r, dt, rng):
    rho = (mu - r) / sigma
    eps = rng.normal(0.0, 1.0)
    return x_t + sigma * u_t * (rho * dt + math.sqrt(dt) * eps)

## 3. EMV : fonctions de base

`V_theta_fn` définit la forme paramétrique de la fonction de valeur $V^\theta(t,x)$.

`emv_td_error` calcule l’erreur de Bellman continue utilisée dans l’étape de policy evaluation.

`grad_theta_fn` calcule les gradients par rapport à $\theta_1$ et $\theta_2$.

`grad_phi_fn` calcule les gradients par rapport à $\phi_1$ et $\phi_2$.

`emv_policy_mean_var` reconstruit la moyenne et la variance de la politique gaussienne à partir de $(\phi_1,\phi_2)$.

In [158]:
def V_theta_fn(theta0, theta1, theta2, theta3, w, T, x, t):
    # Eq. (44) : fonction de valeur paramétrique
    x_arr = np.asarray(x, dtype=float)
    t_arr = np.asarray(t, dtype=float)
    xd = x_arr - w
    return (xd**2) * np.exp(-theta3 * (T - t_arr)) + theta2 * (t_arr**2) + theta1 * t_arr + theta0

def emv_td_error(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                 vec_xi, vec_xi1, vec_ti, vec_ti1):
    # Eq. (42) : erreur de Bellman en temps continu
    vdot = (
        V_theta_fn(theta0, theta1, theta2, theta3, w, T, vec_xi1, vec_ti1)
        - V_theta_fn(theta0, theta1, theta2, theta3, w, T, vec_xi, vec_ti)
    ) / dt
    return vdot - lam * (phi1 + phi2 * (T - vec_ti))

def grad_theta_fn(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                  vec_xi, vec_xi1, vec_ti, vec_ti1):
    # Eq. (47) et (48) : gradients en theta1 et theta2
    td = emv_td_error(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                      vec_xi, vec_xi1, vec_ti, vec_ti1)
    g1 = float(np.sum(td * dt))
    g2 = float(np.sum(td * (vec_ti1**2 - vec_ti**2)))
    return g1, g2

def grad_phi_fn(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                vec_xi, vec_xi1, vec_ti, vec_ti1):
    # Eq. (49) et (50) : gradients en phi1 et phi2
    td = emv_td_error(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                      vec_xi, vec_xi1, vec_ti, vec_ti1)

    x1 = vec_xi1 - w
    x0 = vec_xi - w

    f1 = 2 * (x1**2) * np.exp(-2 * phi2 * (T - vec_ti1)) * (T - vec_ti1)
    f2 = 2 * (x0**2) * np.exp(-2 * phi2 * (T - vec_ti)) * (T - vec_ti)
    sf = -(f1 - f2) / dt - lam * (T - vec_ti)

    gp1 = -lam * float(np.sum(td * dt))
    gp2 = float(np.sum(td * dt * sf))
    return gp1, gp2

def emv_policy_mean_var(phi1, phi2, x, w, t, T, lam, rho_sign):
    # Eq. (46) : moyenne et variance de la politique gaussienne
    sp2 = float(np.clip(phi2, *PHI2_CLIP))

    # Calcul de la moyenne en log-espace pour éviter les overflow
    log_mean_coeff = 0.5 * math.log(2 * sp2 / (lam * math.pi)) + 0.5 * (2 * phi1 - 1)
    log_mean_coeff = float(np.clip(log_mean_coeff, -50.0, 50.0))
    mean_coeff = -rho_sign * math.exp(log_mean_coeff)
    mean = mean_coeff * (x - w)

    # Calcul de la variance en log-espace pour éviter les overflow
    log_var = math.log(1 / (2 * math.pi)) + 2 * sp2 * (T - t) + 2 * phi1 - 1
    log_var = float(np.clip(log_var, *LOG_VAR_CLIP))
    var = math.exp(log_var)

    return float(mean), float(var)

In [159]:
def grad_theta_step(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                    x_i, x_ip1, t_i, t_ip1):
    td = emv_td_error(
        theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
        np.array([x_i]), np.array([x_ip1]), np.array([t_i]), np.array([t_ip1])
    )[0]
    g1 = td * dt
    g2 = td * (t_ip1**2 - t_i**2)
    return g1, g2

def grad_phi_step(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                  x_i, x_ip1, t_i, t_ip1):
    td = emv_td_error(
        theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
        np.array([x_i]), np.array([x_ip1]), np.array([t_i]), np.array([t_ip1])
    )[0]

    gp1 = -lam * td * dt

    f1 = 2 * ((x_ip1 - w)**2) * np.exp(-2 * phi2 * (T - t_ip1)) * (T - t_ip1)
    f0 = 2 * ((x_i - w)**2) * np.exp(-2 * phi2 * (T - t_i)) * (T - t_i)
    sf = -(f1 - f0) / dt - lam * (T - t_i)

    gp2 = td * dt * sf
    return gp1, gp2

## 4. Boucle principale EMV

`run_emv` exécute l’algorithme EMV sur un marché stationnaire.

À chaque épisode, une trajectoire de prix est simulée, puis une suite d’allocations $u_t$ est tirée à partir de la politique gaussienne courante.

Après chaque pas de temps, les paramètres de la fonction de valeur $(\theta_1,\theta_2)$ et ceux de la politique $(\phi_1,\phi_2)$ sont mis à jour.

Le paramètre $w$ est corrigé régulièrement pour rapprocher la moyenne de la richesse terminale de la cible $z$.

In [160]:
def run_emv(mu, sigma, r=0.02, x0=1.0, z=1.4, T=1.0, dt=1.0/252,
            lam=2.0, M=5000, N=10, alpha=0.05, eta_theta=5e-4, eta_phi=5e-4,
            seed=12345):

    nst = int(round(T / dt))
    rho = (mu - r) / sigma
    rho_sign = 1.0 if rho >= 0 else -1.0

    env_rng = np.random.default_rng(seed)
    action_rng = np.random.default_rng(seed + 999)

    # Init gaussienne simple de la forme c1 * exp(c2(T-t))
    c2_init = 1.0

    phi1 = 0.5 * math.log(2.0 * math.pi * math.e * lam)
    
    phi2 = 0.5 * c2_init

    theta1 = 0.0
    theta2 = 0.0
    theta3 = 2.0 * phi2
    w = z
    theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

    xT_list = []
    w_buffer = []
    hist = {k: [] for k in ['phi1', 'phi2', 'theta0', 'theta1', 'theta2', 'theta3', 'w']}

    for k in range(M):
        phi1_ep = phi1
        phi2_ep = phi2
        theta1_ep = theta1
        theta2_ep = theta2
        theta3_ep = theta3
        theta0_ep = theta0
        w_ep = w

        X = [x0]
        times = [0.0]

        for i in range(1, nst + 1):
            t_prev = times[-1]

            mean_u, var_u = emv_policy_mean_var(
                phi1_ep, phi2_ep, X[-1], w_ep, t_prev, T, lam, rho_sign
            )
            u_t = action_rng.normal(mean_u, math.sqrt(max(var_u, 1e-12)))

            x_next = wealth_step_direct(X[-1], u_t, mu, sigma, r, dt, env_rng)
            X.append(float(np.clip(x_next, -1e9, 1e9)))
            times.append(float(i * dt))

        vxi = np.array(X[:-1], dtype=float)
        vxi1 = np.array(X[1:], dtype=float)
        vti = np.array(times[:-1], dtype=float)
        vti1 = np.array(times[1:], dtype=float)

        gt1, gt2 = grad_theta_fn(
            theta0_ep, theta1_ep, theta2_ep, theta3_ep, w_ep, T, dt, lam, phi1_ep, phi2_ep,
            vxi, vxi1, vti, vti1
        )
        gp1, gp2 = grad_phi_fn(
            theta0_ep, theta1_ep, theta2_ep, theta3_ep, w_ep, T, dt, lam, phi1_ep, phi2_ep,
            vxi, vxi1, vti, vti1
        )

        theta1 = theta1_ep - eta_theta * gt1
        theta2 = theta2_ep - eta_theta * gt2
        phi1 = phi1_ep - eta_phi * gp1
        phi2 = phi2_ep - eta_phi * gp2
        phi2 = float(np.clip(phi2, *PHI2_CLIP))

        theta3 = 2.0 * phi2
        theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

        x_T = float(X[-1])
        xT_list.append(x_T)
        w_buffer.append(x_T)

        if (k + 1) % N == 0:
            n_w = (k + 1) // N
            w -= alpha * (float(np.mean(w_buffer)) - z)
            w_buffer = []
            theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

        hist['phi1'].append(phi1)
        hist['phi2'].append(phi2)
        hist['theta0'].append(theta0)
        hist['theta1'].append(theta1)
        hist['theta2'].append(theta2)
        hist['theta3'].append(theta3)
        hist['w'].append(w)

    return {'xT': np.array(xT_list), **{k: np.array(v) for k, v in hist.items()}}

## 6. Test scenario unique : mu=30%, sigma=10%

Comparaison EMV et MLE sur un même cas test, ici $\mu=0.30$ et $\sigma=0.10$. On lance d’abord l’algorithme EMV, puis le benchmark MLE, chacun sur $20,000$ épisodes. Pour chaque méthode, on récupère les richesses terminales $X_T$, puis on calcule sur les $2,000$ derniers épisodes la moyenne, la variance et un Sharpe empirique. Pour EMV, on affiche aussi la valeur finale de $\theta_3$ et de $w$ afin de voir si l’algorithme converge vers les quantités théoriques attendues.


In [161]:
mu_test, sigma_test = -0.50, 0.10
# rho_true = (mu_test - r_rate) / sigma_test

# print("EMV en cours...")
# t0 = time.time()
# emv = run_emv(mu_test, sigma_test, M=20000)
# print(f"EMV termine en {time.time()-t0:.0f}s")
# last = emv['xT'][-2000:]
# me, ve = last.mean(), last.var()
# sre = (me-1)/np.sqrt(ve) if ve > 0 else 0
# print(f"EMV : E[xT]={me:.4f} Var={ve:.4f} SR={sre:.4f}")
# print(f"theta3={emv['theta3'][-1]:.4f} (rho^2={rho_true**2:.4f})")
# print(f"w={emv['w'][-1]:.4f} (w*={compute_w_star(rho_true):.4f})")
# print()


In [162]:
# fig, axes = plt.subplots(2, 3, figsize=(18, 10))
# xT = emv['xT']; w50=50; nw=len(xT)//w50
# means = [xT[i*w50:(i+1)*w50].mean() for i in range(nw)]
# varis = [xT[i*w50:(i+1)*w50].var() for i in range(nw)]
# ep = np.arange(1,nw+1)*w50

# axes[0,0].plot(ep,means,'b-'); axes[0,0].axhline(z,color='r',ls='--')
# axes[0,0].set_title('Moyenne xT (w=50)'); axes[0,0].set_xlabel('episode')

# axes[0,1].semilogy(ep,np.maximum(varis,1e-8),'b-')
# axes[0,1].set_title('Variance xT (log)'); axes[0,1].set_xlabel('episode')

# axes[0,2].plot(emv['w']); axes[0,2].axhline(compute_w_star(rho_true),color='r',ls='--')
# axes[0,2].set_title('w'); axes[0,2].set_xlabel('episode')

# axes[1,0].plot(emv['theta3']); axes[1,0].axhline(rho_true**2,color='r',ls='--')
# axes[1,0].set_title('theta3 (rho^2)'); axes[1,0].set_xlabel('episode')

# axes[1,1].plot(emv['phi1']); axes[1,1].set_title('phi1'); axes[1,1].set_xlabel('episode')

# axes[1,2].plot(emv['phi2']); axes[1,2].axhline(rho_true**2/2,color='r',ls='--')
# axes[1,2].set_title('phi2 (rho^2/2)'); axes[1,2].set_xlabel('episode')

# plt.suptitle(f'EMV mu={mu_test}, sigma={sigma_test}', fontweight='bold')
# plt.tight_layout(); plt.show()

## 7. Grille de lambda constant

In [163]:
def summarize_run(res, tail=2000, lam=None, x_ref=1.0, T=1.0, rho_sign=1.0):
    last = np.asarray(res['xT'][-tail:], dtype=float)
    me = float(last.mean())
    ve = float(last.var())
    sre = (me - 1.0) / math.sqrt(ve) if ve > 0 else np.nan

    policy_var_avg = np.nan
    policy_var_t0 = np.nan
    policy_var_tmid = np.nan
    policy_var_tT = np.nan

    if lam is not None:
        phi1 = float(res['phi1'][-1])
        phi2 = float(res['phi2'][-1])
        w = float(res['w'][-1])

        # variance de policy à quelques dates
        _, policy_var_t0 = emv_policy_mean_var(phi1, phi2, x_ref, w, 0.0, T, lam, rho_sign)
        _, policy_var_tmid = emv_policy_mean_var(phi1, phi2, x_ref, w, 0.5*T, T, lam, rho_sign)
        _, policy_var_tT = emv_policy_mean_var(phi1, phi2, x_ref, w, T, T, lam, rho_sign)

        # moyenne sur une grille de temps
        ts = np.linspace(0.0, T, 50)
        vars_u = []
        for t in ts:
            _, v = emv_policy_mean_var(phi1, phi2, x_ref, w, t, T, lam, rho_sign)
            vars_u.append(v)
        policy_var_avg = float(np.mean(vars_u))

    return {
        'mean_xT': me,
        'var_xT': ve,
        'sr_emp': sre,
        'w_last': float(res['w'][-1]),
        'theta3_last': float(res['theta3'][-1]),
        'phi2_last': float(res['phi2'][-1]),
        'phi1_last': float(res['phi1'][-1]),
        'policy_var_t0': policy_var_t0,
        'policy_var_tmid': policy_var_tmid,
        'policy_var_tT': policy_var_tT,
        'policy_var_avg': policy_var_avg,
    }

def lambda_grid_experiment(mu, sigma, lam_list, seeds, **kwargs):
    rows = []
    rho_true = sharpe_ratio(mu, kwargs.get('r', 0.02), sigma)
    rho_sign = 1.0 if rho_true >= 0 else -1.0
    w_star = compute_w_star(
        rho_true,
        T=kwargs.get('T', 1.0),
        x0=kwargs.get('x0', 1.0),
        z=kwargs.get('z', 1.4)
    )

    for lam in lam_list:
        stats_lam = []
        for seed in seeds:
            res = run_emv(mu, sigma, lam=lam, seed=seed, **kwargs)
            s = summarize_run(
                res,
                tail=2000,
                lam=lam,
                x_ref=kwargs.get('x0', 1.0),
                T=kwargs.get('T', 1.0),
                rho_sign=rho_sign
            )
            stats_lam.append(s)

        row = {'lam': lam}
        for key in stats_lam[0].keys():
            vals = np.array([d[key] for d in stats_lam], dtype=float)
            row[key] = float(np.mean(vals))

        row['rho2_true'] = rho_true**2
        row['w_star'] = w_star
        rows.append(row)

    return pd.DataFrame(rows)

In [164]:
lam_list = [0.5, 1.0, 2.0, 3.0, 5.0]
# lam_list = [2.0]
seeds = [123]
mu_test, sigma_test = -0.30, 0.10

df_lambda = lambda_grid_experiment(
    mu=mu_test,
    sigma=sigma_test,
    lam_list=lam_list,
    seeds=seeds,
    r=r_rate,
    x0=x0,
    z=z,
    T=T,
    dt=dt,
    M=20000,
    N=10,
    alpha=0.05,
    eta_theta=5e-4,
    eta_phi=5e-4,
)

print(df_lambda.round(4))

   lam  mean_xT  var_xT  sr_emp  w_last  theta3_last  phi2_last  phi1_last  \
0  0.5   1.3963  0.0312  2.2430  1.9336       5.1604     2.5802     0.4692   
1  1.0   1.3879  0.0170  2.9759  3.0927       3.5473     1.7736     0.0097   
2  2.0   1.3855  0.0158  3.0619  4.3913       2.6007     1.3004    -0.0165   
3  3.0   1.3850  0.0161  3.0349  5.0422       2.3029     1.1514     0.0530   
4  5.0   1.3843  0.0165  2.9923  5.8982       2.0045     1.0022     0.1694   

   policy_var_t0  policy_var_tmid  policy_var_tT  policy_var_avg  rho2_true  \
0        26.0742           1.9753         0.1496          5.1900      10.24   
1         2.0727           0.3518         0.0597          0.5777      10.24   
2         0.7632           0.2079         0.0566          0.2745      10.24   
3         0.6511           0.2059         0.0651          0.2566      10.24   
4         0.6098           0.2238         0.0822          0.2649      10.24   

   w_star  
0     1.4  
1     1.4  
2     1.4  
3     1.

## 8. Lambda décroissant

In [59]:
def lambda_decay_schedule(k, M, lam0=2.0, c=200.0):
    """
    Schedule du papier (Section 5.3) :
        lambda_k = lam0 * (1 - exp(c * (k - M) / M))
    avec k = 0, 1, ..., M.

    Ici k est l'indice d'épisode courant.
    """
    val = lam0 * (1.0 - math.exp(c * (k - M) / M))
    return max(val, 0.0)

In [60]:
def run_emv_decay(mu, sigma, r=0.02, x0=1.0, z=1.4, T=1.0, dt=1.0/252,
                  lam0=2.0, decay_c=200.0,
                  M=5000, N=10, alpha=0.05, eta_theta=5e-4, eta_phi=5e-4,
                  seed=12345):

    nst = int(round(T / dt))
    rho = (mu - r) / sigma
    rho_sign = 1.0 if rho >= 0 else -1.0

    env_rng = np.random.default_rng(seed)
    action_rng = np.random.default_rng(seed + 999)

    # Init gaussienne simple
    c1_init = 1.0
    c2_init = 1.0

    phi1 = 0.5 * math.log(2.0 * math.pi * math.e * c1_init)
    phi2 = 0.5 * c2_init

    theta1 = 0.0
    theta2 = 0.0
    theta3 = 2.0 * phi2
    w = z
    theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

    xT_list = []
    w_buffer = []

    hist = {
        'phi1': [], 'phi2': [], 'theta0': [], 'theta1': [],
        'theta2': [], 'theta3': [], 'w': [], 'lam': []
    }

    for k in range(M):
        lam_k = lambda_decay_schedule(k, M=M, lam0=lam0, c=decay_c)

        phi1_ep = phi1
        phi2_ep = phi2
        theta1_ep = theta1
        theta2_ep = theta2
        theta3_ep = theta3
        theta0_ep = theta0
        w_ep = w

        X = [x0]
        times = [0.0]

        for i in range(1, nst + 1):
            t_prev = times[-1]

            mean_u, var_u = emv_policy_mean_var(
                phi1_ep, phi2_ep, X[-1], w_ep, t_prev, T, lam_k, rho_sign
            )
            u_t = action_rng.normal(mean_u, math.sqrt(max(var_u, 1e-12)))

            x_next = wealth_step_direct(X[-1], u_t, mu, sigma, r, dt, env_rng)
            X.append(float(np.clip(x_next, -1e9, 1e9)))
            times.append(float(i * dt))

        vxi = np.array(X[:-1], dtype=float)
        vxi1 = np.array(X[1:], dtype=float)
        vti = np.array(times[:-1], dtype=float)
        vti1 = np.array(times[1:], dtype=float)

        gt1, gt2 = grad_theta_fn(
            theta0_ep, theta1_ep, theta2_ep, theta3_ep, w_ep, T, dt, lam_k, phi1_ep, phi2_ep,
            vxi, vxi1, vti, vti1
        )
        gp1, gp2 = grad_phi_fn(
            theta0_ep, theta1_ep, theta2_ep, theta3_ep, w_ep, T, dt, lam_k, phi1_ep, phi2_ep,
            vxi, vxi1, vti, vti1
        )

        eta_theta_k = eta_theta / math.sqrt(k + 1.0)
        eta_phi_k = eta_phi / math.sqrt(k + 1.0)

        theta1 = theta1_ep - eta_theta_k * gt1
        theta2 = theta2_ep - eta_theta_k * gt2
        phi1 = phi1_ep - eta_phi_k * gp1
        phi2 = phi2_ep - eta_phi_k * gp2
        phi2 = float(np.clip(phi2, *PHI2_CLIP))

        theta3 = 2.0 * phi2
        theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

        x_T = float(X[-1])
        xT_list.append(x_T)
        w_buffer.append(x_T)

        if (k + 1) % N == 0:
            n_w = (k + 1) // N
            alpha_k = alpha / math.sqrt(n_w)
            w -= alpha_k * (float(np.mean(w_buffer)) - z)
            w_buffer = []
            theta0 = -theta2 * T**2 - theta1 * T - (w - z)**2

        hist['phi1'].append(phi1)
        hist['phi2'].append(phi2)
        hist['theta0'].append(theta0)
        hist['theta1'].append(theta1)
        hist['theta2'].append(theta2)
        hist['theta3'].append(theta3)
        hist['w'].append(w)
        hist['lam'].append(lam_k)

    return {'xT': np.array(xT_list), **{k: np.array(v) for k, v in hist.items()}}

In [ ]:
def test_decay_grid(mu, sigma, decay_c_list, seed=123, lam0=2.0,
                    r=0.02, x0=1.0, z=1.4, T=1.0, dt=1.0/252,
                    M=5000, N=10, alpha=0.05, eta_theta=5e-4, eta_phi=5e-4,
                    tail=2000):
    rows = []

    rho_true = sharpe_ratio(mu, r, sigma)
    rho_sign = 1.0 if rho_true >= 0 else -1.0

    for c in decay_c_list:
        res = run_emv_decay(
            mu=mu,
            sigma=sigma,
            r=r,
            x0=x0,
            z=z,
            T=T,
            dt=dt,
            lam0=lam0,
            decay_c=c,
            M=M,
            N=N,
            alpha=alpha,
            eta_theta=eta_theta,
            eta_phi=eta_phi,
            seed=seed
        )

        last = np.asarray(res['xT'][-tail:], dtype=float)
        me = float(last.mean())
        ve = float(last.var())
        sre = (me - 1.0) / math.sqrt(ve) if ve > 0 else np.nan

        phi1 = float(res['phi1'][-1])
        phi2 = float(res['phi2'][-1])
        w = float(res['w'][-1])

        # variance de la policy à quelques dates
        _, policy_var_t0 = emv_policy_mean_var(phi1, phi2, x0, w, 0.0, T, lam0, rho_sign)
        _, policy_var_tmid = emv_policy_mean_var(phi1, phi2, x0, w, 0.5*T, T, lam0, rho_sign)
        _, policy_var_tT = emv_policy_mean_var(phi1, phi2, x0, w, T, T, lam0, rho_sign)

        # moyenne de la variance de policy sur le temps
        ts = np.linspace(0.0, T, 50)
        lam_path = res['lam']
        vars_u = []
        for j, t in enumerate(ts):
            k_idx = min(int(j * (len(lam_path)-1) / (len(ts)-1)), len(lam_path)-1)
            lam_t = float(lam_path[k_idx])
            _, v = emv_policy_mean_var(phi1, phi2, x0, w, t, T, lam_t, rho_sign)
            vars_u.append(v)
        policy_var_avg = float(np.mean(vars_u))

        row = {
            "decay_c": c,
            "mean_xT": me,
            "var_xT": ve,
            "sr_emp": sre,
            "w_last": float(res['w'][-1]),
            "theta3_last": float(res['theta3'][-1]),
            "phi2_last": float(res['phi2'][-1]),
            "policy_var_t0": policy_var_t0,
            "policy_var_tmid": policy_var_tmid,
            "policy_var_tT": policy_var_tT,
            "policy_var_avg": policy_var_avg,
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
decay_c_list = [5, 20, 200, 400]

df_decay = test_decay_grid(
    mu=mu_test,
    sigma=sigma_test,
    decay_c_list=decay_c_list,
    seed=123,
    lam0=2.0,
    r=r_rate,
    x0=x0,
    z=z,
    T=T,
    dt=dt,
    M=5000,
    N=10,
    alpha=0.05,
    eta_theta=5e-4,
    eta_phi=5e-4,
    tail=2000
)

print(df_decay.round(4))

KeyboardInterrupt: 